# Demo 2: Search and Context with Actor Identity
This notebook teaches how to move from simple memory writes to context-rich retrieval using the Ninai SDK.

## Learning Goals
1. Authenticate with `NinaiClient`.
2. Insert related operational notes.
3. Run hybrid search and inspect contextual fields.
4. Understand why relevance is more than just score.

## What You Should Expect
- You will see `search_count` greater than `0`.
- Each result should include business context fields like scope, classification, source, timestamps, and tags.
- Scores may look numerically small; this is normal for hybrid ranking.

## Step 1: Setup and Login

Run the next cell to initialize the SDK and authenticate.

Expected outcome:
- No errors.
- A `client` object is ready.
- A unique `seed` value is created to isolate this run from older notebook data.

In [1]:
from ninai import NinaiClient
import uuid

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

## Step 2: Write Notes and Retrieve Context

Run the next cell to:
1. Insert three related notes with tags.
2. Search using `hybrid=True`.
3. Print context-rich fields for each result.

Expected outcome:
- `search_count` is usually between 3 and 10.
- Result rows include fields such as `scope`, `classification`, `source_type`, `created_at`, and `content_preview`.
- `latest_seed_hits` should be at least 3 for the current run.

In [3]:
notes = [
    f'ACME login issue after SSO cutover seed={seed}',
    f'Password reset did not resolve ACME issue seed={seed}',
    f'Engineering suspects token audience mismatch seed={seed}',
]

for note in notes:
    client.memories.create(content=note, source_type='manual', tags=['acme', 'auth', seed])

results = client.memories.search(query=f'ACME {seed}', limit=10, hybrid=True)
print('seed:', seed)
print('search_count:', len(results.items))

if not results.items:
    print('No search hits yet. This can happen briefly while indexing catches up.')

for idx, mem in enumerate(results.items, 1):
    print(f"\n--- Result {idx} ---")
    print('id:', mem.id)
    print('score:', mem.score)
    print('scope:', mem.scope, '| scope_id:', mem.scope_id)
    print('classification:', mem.classification, '| required_clearance:', mem.required_clearance)
    print('source_type:', mem.source_type)
    print('organization_id:', mem.organization_id, '| owner_id:', mem.owner_id)
    print('tags:', mem.tags)
    print('created_at:', mem.created_at, '| updated_at:', mem.updated_at)
    print('access_count:', mem.access_count, '| last_accessed_at:', mem.last_accessed_at)
    print('is_promoted:', mem.is_promoted)
    print('content_preview:', mem.content_preview)

latest = client.memories.list(page_size=10)
seed_hits = [m for m in latest.items if (m.content_preview and seed in m.content_preview) or (seed in ' '.join(m.tags or []))]
print(f"\nlatest_list_count: {len(latest.items)} | latest_seed_hits: {len(seed_hits)}")

seed: b0e0d528
search_count: 5

--- Result 1 ---
id: b2d63ad7-43b5-4178-a369-03a6826c8bc4
score: 0.3
scope: personal | scope_id: None
classification: internal | required_clearance: 0
source_type: manual
organization_id: 550e8400-e29b-41d4-a716-446655440000 | owner_id: 550e8400-e29b-41d4-a716-446655440100
tags: ['acme', 'auth', 'b0e0d528']
created_at: 2026-04-14 15:12:50.612699+00:00 | updated_at: 2026-04-14 15:12:50.612704+00:00
access_count: 0 | last_accessed_at: None
is_promoted: False
content_preview: Password reset did not resolve ACME issue seed=b0e0d528

--- Result 2 ---
id: 4875acaa-c7d5-4ad6-b1ae-c5b27a663b05
score: 0.27749064747541513
scope: personal | scope_id: None
classification: internal | required_clearance: 0
source_type: manual
organization_id: 550e8400-e29b-41d4-a716-446655440000 | owner_id: 550e8400-e29b-41d4-a716-446655440100
tags: ['acme', 'auth', 'b0e0d528']
created_at: 2026-04-14 15:12:50.415526+00:00 | updated_at: 2026-04-14 15:12:50.415535+00:00
access_count: 0 

## Step 3: How to Interpret Results

Use this checklist after running the previous cell:

### Relevance Check
- Top results should mention ACME and the current `seed`.
- If older runs appear, compare tags and timestamps.

### Context Check
- Verify governance fields: `scope`, `classification`, `required_clearance`.
- Verify provenance fields: `source_type`, `organization_id`, `owner_id`.
- Verify lifecycle fields: `created_at`, `updated_at`, `access_count`.

### About Score
- Lower numeric score does not mean poor relevance in hybrid search.
- Focus on rank order + contextual correctness rather than score alone.

### If Results Look Wrong
1. Re-run the setup cell to generate a fresh `seed`.
2. Re-run the search cell.
3. Increase `limit` to inspect more candidates.